# CELLxGENE Census PBMC → linked Landscape–Clustergram

**First draft for review; not part of the documentation.** Query healthy peripheral blood mononuclear cells (PBMCs), download a small AnnData, and link a UMAP Landscape to a cell-type marker dot plot.

This follows the [original Colab draft](https://colab.research.google.com/drive/1KBWBVhUHIMEKQ1d21UqmyJuyDVh9WU34), using the Census convenience API instead of manually managing SOMA axis queries. We keep its `2025-01-30` Census snapshot for reproducibility. The starter dataset is [Hao et al. (2021), Integrated analysis of multimodal single-cell data](https://doi.org/10.1016/j.cell.2021.04.048): one healthy donor, at most 5,000 cells, and published cell-type annotations.

Run top to bottom in **JupyterLab with a local Python 3.11/3.12 kernel**, using this checkout's `celldega` version. The Landscape reads local files over HTTP; keep the kernel running. Downloads, processed AnnDatas, and generated Landscape files stay in the Git-ignored `notebooks/data/cellxgene_census/` directory.

API references: [querying metadata](https://chanzuckerberg.github.io/cellxgene-census/_autosummary/cellxgene_census.get_obs.html), [fetching AnnData](https://chanzuckerberg.github.io/cellxgene-census/_autosummary/cellxgene_census.get_anndata.html), and [Census schema](https://chanzuckerberg.github.io/cellxgene-census/cellxgene_census_docsite_schema.html).

## 1. Setup

**Required tutorial dependency:** `cellxgene-census>=1.18,<2` is an extra package and is not included with Celldega. Run the next cell once to install it into the notebook’s active kernel environment. Select the kernel backed by the repository’s `dega` environment. From the repository root, install this branch with `python -m pip install -e ".[scanpy]"`. Restart the kernel after installing packages, then run from the top. No API key is needed for public Census data.

This notebook uses the **local, unpublished front-end bundle** at `src/celldega/static/celldega.js`. Run your build watch from the repository root (`npm run watch`); the setup cell enables Anywidget hot reload so rebuilt bundles refresh the widgets. **Restart the kernel once and run from the top** if Celldega or Anywidget was already imported before these environment settings were applied.

In [ ]:
# Required tutorial dependency: run once in this notebook’s active kernel.
%pip install "cellxgene-census>=1.18,<2"

In [ ]:
import os

# Set these before importing Celldega or Anywidget.
os.environ["CELLDEGA_LOCAL_ESM"] = "1"
os.environ["ANYWIDGET_HMR"] = "1"

from importlib.metadata import version
from pathlib import Path
import hashlib
import json
from urllib.parse import quote

import anndata as ad
import cellxgene_census as census_api
from IPython.display import display
import numpy as np
import pandas as pd
import scanpy as sc

import celldega as dega

print({name: version(name) for name in ("celldega", "cellxgene-census", "scanpy")})
frontend_bundle = Path(dega.__file__).resolve().parent / "static" / "celldega.js"
if not frontend_bundle.is_file():
    raise FileNotFoundError("Build the local front-end first with npm run build or npm run watch.")
print(f"Local front-end: {frontend_bundle}")
sc.settings.verbosity = 1

## 2. Choose a bounded query

The Census contains repeated cells across some datasets; `is_primary_data` excludes those copies. Keep one dataset and one donor for this first example so we can explore without batch integration. Sampling is seeded and happens **before** fetching the expression matrix. All genes are fetched so normalization uses each cell's full library, rather than only a marker panel.

Change the settings here to fetch a different slice. Each query gets its own cache directory.

In [ ]:
CENSUS_VERSION = "2025-01-30"
ORGANISM = "Homo sapiens"
DATASET_ID = "ed5d841d-6346-47d4-ab2f-7119ad7e3a35"  # nygc multimodal pbmc
CELL_FILTER = "is_primary_data == True and disease == 'normal' and donor_id == 'P8'"
MAX_CELLS = 5_000
SEED = 0
GROUPBY = "cell_type"
MIN_CELLS_PER_GENE = 3  # retain the filtered transcriptome in the all view

query_config = dict(
    census_version=CENSUS_VERSION, organism=ORGANISM, dataset_id=DATASET_ID,
    cell_filter=CELL_FILTER, max_cells=MAX_CELLS, seed=SEED,
)
cache_key = hashlib.sha256(json.dumps(query_config, sort_keys=True).encode()).hexdigest()[:12]
cwd = Path.cwd().resolve()
project_root = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir()),
    None,
)
data_root = (project_root / "notebooks" if project_root else cwd) / "data" / "cellxgene_census"
data_dir = data_root / cache_key
data_dir.mkdir(parents=True, exist_ok=True)
raw_path = data_dir / "pbmc_raw.h5ad"
prepared_path = data_dir / "pbmc_prepared.h5ad"
print(data_dir)

In [ ]:
if raw_path.is_file():
    adata = ad.read_h5ad(raw_path)
    print(f"Reusing {raw_path.name}")
else:
    obs_filter = f"dataset_id == '{DATASET_ID}' and ({CELL_FILTER})"
    with census_api.open_soma(census_version=CENSUS_VERSION) as census:
        source = census["census_info"]["datasets"].read(
            value_filter=f"dataset_id == '{DATASET_ID}'"
        ).concat().to_pandas()
        if source.empty:
            raise ValueError("Dataset not found in this Census snapshot; choose another dataset/version.")

        # Read only a few metadata columns, then choose cell coordinates.
        eligible = census_api.get_obs(
            census, ORGANISM, value_filter=obs_filter,
            column_names=["soma_joinid", "cell_type", "donor_id", "assay", "tissue", "disease"],
        ).sort_values("soma_joinid")
        if eligible.empty:
            raise ValueError("No matching cells; inspect CELL_FILTER and DATASET_ID.")
        selected = eligible.sample(n=min(MAX_CELLS, len(eligible)), random_state=SEED)
        cell_ids = np.sort(selected["soma_joinid"].to_numpy(dtype=np.int64))
        print(f"Fetching {len(cell_ids):,} of {len(eligible):,} eligible cells")
        adata = census_api.get_anndata(
            census, ORGANISM, X_name="raw", obs_coords=cell_ids,
            obs_column_names=["soma_joinid", "dataset_id", "cell_type", "cell_type_ontology_term_id",
                              "donor_id", "assay", "tissue", "disease", "is_primary_data"],
            var_column_names=["feature_id", "feature_name"],
        )
    # Remove unused Census-wide categories before saving/displaying the small subset.
    for column in adata.obs.select_dtypes(include="category"):
        adata.obs[column] = adata.obs[column].cat.remove_unused_categories()
    adata.uns["census_query"] = query_config
    adata.uns["dataset_citation"] = str(source.iloc[0]["citation"])
    source.to_json(data_dir / "source_dataset.json", orient="records", indent=2)
    partial = raw_path.with_name("pbmc_raw.part.h5ad")
    adata.write_h5ad(partial, compression="gzip")
    partial.replace(raw_path)

print(adata)
print(adata.uns["dataset_citation"])
display(adata.obs[GROUPBY].value_counts().rename("downloaded_cells").to_frame())

## 3. Prepare expression and UMAP

Census `X_name="raw"` supplies counts, not a log-normalized matrix. Keep those counts for Landscape gene overlays; normalize to 10,000 counts per cell and apply `log1p` for PCA and the marker dot plot. Preserve Ensembl IDs in `var['feature_id']` while using unique gene symbols for display.

The source is already curated. These light filters remove empty/very sparse cells and genes, plus cell types represented by fewer than five sampled cells. We use the published cell types directly, rather than adding a new Leiden clustering. The UMAP is recomputed from this subset and is **not a tissue coordinate map**.

In [ ]:
adata.var_names = pd.Index(adata.var["feature_name"].astype(str))
adata.var_names_make_unique()
adata.obs_names = pd.Index(adata.obs["soma_joinid"].astype(str))
adata.obs_names.name = None
adata.X = adata.X.tocsr().astype(np.float32)

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
labels = adata.obs[GROUPBY].astype(str)
counts = labels.value_counts()
keep = labels.map(counts).ge(5) & ~labels.isin(["unknown", "na", "nan"])
adata = adata[keep].copy()
# Reapply after dropping rare cell types so every retained gene is detected.
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
adata.obs[GROUPBY] = pd.Categorical(adata.obs[GROUPBY].astype(str))
if adata.obs[GROUPBY].nunique() < 2:
    raise ValueError("Need at least two populated cell types; increase MAX_CELLS or change the query.")

adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=10_000)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=min(2_000, adata.n_vars))
sc.tl.pca(adata, mask_var="highly_variable", svd_solver="arpack", random_state=SEED)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=min(30, adata.obsm["X_pca"].shape[1]), random_state=SEED)
sc.tl.umap(adata, random_state=SEED)
sc.pl.umap(adata, color=GROUPBY, legend_loc="right margin", frameon=False)
print(f"Prepared {adata.n_obs:,} cells × {adata.n_vars:,} genes")

## 4. Build transcriptome-wide cell-type signatures

`Matrix(..., groupby=...)` aggregates every retained gene by cell type: dot color is the gene-wise Z-score of mean log-normalized expression, and dot size is the fraction of cells expressing the gene. **All** contains the complete filtered transcriptome; the marker-count slider provides smaller views. We start at three markers per type for readability.

Scanpy builds its marker-results DataFrame by inserting columns, which produces fragmentation warnings with many groups. Rank ten cell types at a time and assemble the result fields once; each test still compares that cell type against all other cells. No warnings are suppressed, and no installed Scanpy code is modified. These exploratory marker ranks are not donor-level differential-expression evidence.

Full gene clustering uses Euclidean distance on the Z-scored signatures and may take longer than the marker-only draft.

In [ ]:
def rank_markers_in_batches(adata, groupby, n_genes=100, batch_size=10):
    """Avoid Scanpy's fragmented wide table; preserve one-versus-rest tests."""
    groups = adata.obs[groupby].cat.categories.astype(str).tolist()
    parts = []
    for start in range(0, len(groups), batch_size):
        sc.tl.rank_genes_groups(
            adata, groupby=groupby, groups=groups[start:start + batch_size],
            reference="rest", method="wilcoxon", use_raw=False,
            n_genes=min(n_genes, adata.n_vars), tie_correct=True,
        )
        parts.append(adata.uns["rank_genes_groups"])
    combined = {"params": parts[0]["params"].copy()}
    for field in ("names", "scores", "logfoldchanges", "pvals", "pvals_adj"):
        columns = {name: part[field][name] for part in parts for name in part[field].dtype.names}
        combined[field] = np.rec.fromarrays([columns[name] for name in groups], names=groups)
    adata.uns["rank_genes_groups"] = combined


rank_markers_in_batches(adata, GROUPBY)
markers = sc.get.rank_genes_groups_df(adata, group=None)
# Keep all retained genes in the backing matrix and the Landscape export.
view_adata = adata
genes = adata.var_names.tolist()
mat = dega.clust.Matrix(adata, groupby=GROUPBY, col_attr=[GROUPBY], name="census_pbmc")
mat.norm("row", by="zscore")
mat.cluster(dist_type="euclidean", view="rank_genes_groups", levels=[1, 3, 5, 10, 25, 50])
cgm = dega.viz.Clustergram(
    matrix=mat, width=470, height=500, viz_mode="dotplot", rank_dim=3,
)

adata.uns["example_description"] = "Census PBMC subset; full filtered transcriptome; published cell types"
adata.write_h5ad(prepared_path, compression="gzip")
display(markers.groupby("group", observed=True).head(3)[["group", "names", "scores"]])
print(f"All view: {mat.data.shape[0]:,} genes × {mat.data.shape[1]} cell types")
print(f"Prepared AnnData saved to {prepared_path}")

## 5. Generate full-transcriptome DegaFiles

Export raw counts for **every retained gene**, not just the marker panel. DegaFiles store the cell-by-gene matrix as one sparse Parquet file per gene under `cbg/`; the prepared AnnData also keeps the full matrix. The Landscape fetches the requested gene on demand. Re-running this cell regenerates the files and can take a minute for the full transcriptome.

`get_local_server()` serves the current working directory. Launch Jupyter from the repository root or `notebooks/` so the data directory is beneath it. A remote/Colab kernel needs a forwarded HTTP URL for `base_url`.

In [ ]:
landscape_dir = data_dir / "landscape"
dega.pre.make_chromium_from_anndata(view_adata, landscape_dir)
relative_landscape = landscape_dir.relative_to(Path.cwd().resolve()).as_posix()
# Reuse the same server if this cell is run again in the same kernel.
if "landscape_port" not in globals():
    landscape_port = dega.viz.get_local_server()
base_url = f"http://localhost:{landscape_port}/{quote(relative_landscape)}"

## 6. Display the linked Landscape–Clustergram

The Landscape retains individual cells; the Clustergram summarizes cell types.

In [ ]:
landscape = dega.viz.Landscape(
    technology="Chromium", base_url=base_url,
    dataset_name="CELLxGENE · healthy PBMC · donor P8",
    adata=view_adata, cell_attr=[GROUPBY, "donor_id"], cluster_attr=GROUPBY,
    transform=np.eye(3), height=650, nbhd_edit=True,
    ini_zoom=-4.5, ini_x=5_000, ini_y=5_000,
)
gates_path = data_dir / "pbmc_gates.geojson"
if gates_path.exists():
    landscape.load_nbhds(gates_path)

linked = dega.viz.spatial_clustergram(landscape, cgm, width="600px", height="700px")
linked

Try these interactions:

- Click a **gene row** such as `MS4A1`, `CD3D`, or `NKG7` to overlay its raw counts on the UMAP.
- Click a **cell-type column** to highlight the corresponding cells in the Landscape.
- Use the Clustergram marker-count control to switch between 1–50 markers per cell type, or **all** retained genes.
- Compare the cell-type pattern in the UMAP with the marker dot plot. The UMAP shows individual cells; the Clustergram summarizes annotated groups.

### XY expression and NBHD drawing

Select two genes (Shift-click the gene bars, or type `CD3D, MS4A1` in the gene search) and click **XY** to plot them against each other; clicking **XY** with no genes selected starts with overlapping cells, and the **X** and **Y** inputs autocomplete gene names. While XY is active, **LIN** and **LOG** select raw counts versus `log(1 + count)`; positions animate and **UMAP** always returns to the embedding. The small X/Y labels show the genes and scale. This is exploratory gating, without jitter or compensation.

Use **NBHD → SKTCH** to draw a polygon, finish it by double-clicking, and give it a name/color. Polygon membership is calculated on the front end. Drawing or clicking an NBHD selects its cells; they remain blue when switching back to **CELL** or another coordinate space. Cells stay visible as context while drawing. Gates retain their polygon, cell IDs, embedding/sample identity, or ordered gene pair and expression scale. Only gates matching the current space are drawn. Use the methods below to save them in the ignored data directory and restore them in a new session. The same NBHD flow also works in physical space for spatial datasets. `landscape.nbhd_polygons` exposes every saved polygon with its space, cell IDs and x/y vertex rings; physical polygons are converted back to native tissue coordinates, while UMAP/XY retain their displayed coordinates and axis metadata. HTML supports drawing and selection, while the Python save/subset methods require a live notebook kernel.

In [ ]:
# Run after selecting cells or drawing a gate.
selected_ids = landscape.get_selected_cells()  # also available as landscape.selected_cells
selected_adata = landscape.get_selected_cells(adata)
print(f"Selected {len(selected_ids):,} cells; subset shape: {selected_adata.shape}")
if landscape.gates.get("features"):
    landscape.save_nbhds(gates_path)
    print(f"Saved {len(landscape.gates['features'])} gates to {gates_path}")

# Other controls:
# landscape.set_xy("CD3D", "MS4A1", scale="log1p")
# landscape.landscape_state = "umap"
# landscape.load_nbhds(gates_path)
# landscape.select_nbhd("my gate")

# Includes NBHDs from physical, UMAP and XY space; x/y contain polygon rings.
landscape.nbhd_polygons[["name", "space", "x", "y"]]

## Next experiment: mouse brain against a spatial dataset

The same Census query can select `Mus musculus` brain cells, and `var_value_filter="feature_name in ['Gad1', 'Slc17a7', 'Pvalb', 'Sst', 'Vip']"` can request a small gene panel without constructing thousands of OR clauses. Keep a full-gene count matrix for library normalization; subset to the spatial gene panel afterward.

The optional cell below only discovers candidate source datasets; it does not download expression data. Pick a reference matching the spatial sample's brain region, age, assay, and cell/nucleus preparation. Set its `DATASET_ID`, `ORGANISM`, and `CELL_FILTER` above, then repeat the bounded download. A useful next comparison is shared marker expression and cell-type signatures between this reference and an existing mouse-brain MERSCOPE/Visium/MERFISH AnnData. Keep their observation IDs and coordinate systems separate; this draft does not implement label transfer, integration, or spatial alignment.

In [ ]:
# Optional metadata-only discovery; uncomment when ready for the brain follow-up.
# with census_api.open_soma(census_version=CENSUS_VERSION) as census:
#     brain_obs = census_api.get_obs(
#         census, "Mus musculus",
#         value_filter="is_primary_data == True and tissue_general == 'brain' and disease == 'normal'",
#         column_names=["dataset_id", "tissue", "assay", "suspension_type"],
#     )
#     datasets = census["census_info"]["datasets"].read().concat().to_pandas()
# brain_counts = brain_obs.groupby("dataset_id", observed=True).size().rename("matching_cells")
# display(
#     datasets.merge(brain_counts, on="dataset_id")
#     .sort_values("matching_cells", ascending=False)
#     [["dataset_id", "dataset_title", "collection_name", "matching_cells"]].head(15)
# )
# display(brain_obs.groupby(["tissue", "assay", "suspension_type"], observed=True).size()
#         .sort_values(ascending=False).head(15))